# Extremal Coverage Experiments

This notebook organizes the MANY different coverage experiments that were performed in this project for the extremal method, testing out a variety of different procedures and techniques. We attempt to organize these experiments by the choices made, such that we can create a little library of simulation results that we can easily refer to later as we refine our procedure.

Each experiment will fall into one of four buckets: oracle transform, estimated transform, oracle no transform, and estimated no transform. This is essentially to distinguish results from performing a marginal transformation from those without, and further categorizing based on how many oracle assumptions are made. The exact details for each simulation, including sample sizes, gammas, etc. are provided alongside each experiment's result.

At the juncture at which I'm making this notebook, I'm exploring a promising radial bias correction procedure that I have been implementing in the untransformed case, and I imagine I will soon begin exploring its applicability to the transformed case, and so I would like to organize my library of results for when I need to call back on them with ease.

Unless otherwise stated, all simulations are performed on data drawn from a bivariate t distribution with 4 degrees of freedom. Unit variance and 0.7 covariance. Also, all transformations are to a Pareto(1) distribution with lower bound of support at 0.

In [3]:
library(dplyr)
library(ggplot2)
library(readr)
library(mvtnorm)

source('~/isolines_uq/scripts/R/confidence_regions/modules/utils.R')

## 1a. Transformed
A marginal transformation was performed as part of the procedure.

### 2a. Oracle

The marginal distributions were assumed to be known.

**Experiment 1a2a1**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *marginals*: univariate t with 4 df 
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: yes; attempts to construct larger regions over which to take the supremum, exploiting the fact that regular variation bias for probability estimation is worse the sooner you make the assumption. We compute the true relative biases of the regular variation procedure over the true isoline, and use that to find worst-case oracle probability bounds, and then use that to enlarge the region we take suprema and infima over.
+ *simulation script*: `knownmargs_oracleregion_asymmetric_strategy1.R`
+ *tube function*: `computeExtremeRegionBC_GivenBounds`



In [4]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_true_bias_knownmarginals_strategy1.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.984,0.987,0
5000,0.01,0.984,0.987,0
10000,0.01,0.994,0.997,0
50000,0.01,0.999,1.000,0
1000,0.05,0.960,0.963,0
5000,0.05,0.968,0.971,0
10000,0.05,0.976,0.979,0
50000,0.05,0.978,0.981,0
1000,0.10,0.926,0.929,0


**Experiment 1a2a2**
+ *gamma*: 2/3
+ $p_{n}$: 15/n
+ *marginals*: univariate t with 4 df 
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: yes; attempts to construct larger regions over which to take the supremum, exploiting the fact that regular variation bias for probability estimation is worse the sooner you make the assumption. We compute the true relative biases of the regular variation procedure over the true isoline, and use that to find worst-case oracle probability bounds, and then use that to enlarge the region we take suprema and infima over. Specifically, we also bake in the transformation into this computation, sampling datasets, fitting and estimating marginals, and then only using a third of the dataset to actually get the true probability biases.
+ *simulation script*: `knownmargs_oracleregion_asymmetric_strategy3.2.R`
+ *tube function*: `computeExtremeRegionBCThreeway_GivenBounds`



In [5]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_oraclebias_knownmarginals_strategy3.2.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.956,0.959,0
15000,0.01,0.962,0.965,0
30000,0.01,0.948,0.951,0
150000,0.01,0.960,0.963,0
3000,0.05,0.904,0.908,0
15000,0.05,0.896,0.900,0
30000,0.05,0.902,0.906,0
150000,0.05,0.916,0.919,0
3000,0.10,0.846,0.850,0


### 2b. Estimated

The marginal distributions were estimated somehow.

**Experiment 1a2b1**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: none
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations.R`
+ *tube function*: `compute_extreme_region` (function has since been updated with asymmetric tube procedure, which is what we are going with in the future)

In [6]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.992,0.995,0.998
5000,0.01,0.978,0.981,0.984
10000,0.01,0.954,0.957,0.884
50000,0.01,0.894,0.898,0.096
1000,0.05,0.952,0.955,0.886
5000,0.05,0.906,0.910,0.476
10000,0.05,0.842,0.846,0.196
50000,0.05,0.730,0.734,0.000
1000,0.10,0.892,0.896,0.524


**Experiment 1a2b2**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *marginal estimates*: gaussian kde below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: none
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations.R`
+ *tube function*: `compute_extreme_region`

In [7]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_kde_margs.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.
Warning message in qbeta(alpha/2, res$num_covered, 500 - res$num_covered + 1):
“NaNs produced”
Warning message in qbeta(1 - (alpha/2), res$num_covered + 1, 500 - res$num_covered):
“NaNs produced”


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,NaN,NaN,0.999
5000,0.01,0.986,0.989,0.998
10000,0.01,0.964,0.967,0.930
50000,0.01,0.910,0.914,0.120
1000,0.05,NaN,NaN,0.940
5000,0.05,0.932,0.935,0.636
10000,0.05,0.886,0.890,0.260
50000,0.05,0.750,0.754,0.002
1000,0.10,NaN,NaN,0.710


**Experiment 1a2b3**
+ *gamma*: 2/3
+ $p_{n}:$ 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above (**Note:** in bootstrap loop, marginals were refit and transformed on each iteration using the bootstrap sample, but using empirical rank transform instead ($(n/(n+1))\hat{F}_{ecdf}$)
+ *sample splitting procedure*: none
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_bootstrapped_marginals.R`
+ *tube function*: `computeExtremeRegionPretransform`

In [8]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_bootstrapped_marginals.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.700,0.704,0.188
5000,0.01,0.628,0.632,0.024
10000,0.01,0.646,0.650,0.012
50000,0.01,0.510,0.514,0.000
1000,0.05,0.522,0.526,0.038
5000,0.05,0.438,0.442,0.000
10000,0.05,0.454,0.458,0.000
50000,0.05,0.338,0.342,0.000
1000,0.10,0.414,0.418,0.010


**Experiment 1a2b4**
+ *gamma*: 2/3
+ $p_{n}:$ 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above (**Note:** in bootstrap loop, marginals were refit and transformed on each iteration using the bootstrap sample, using the same ecdf/GPD blend)
+ *sample splitting procedure*: none
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_bootstrapped_marginals.R`
+ *tube function*: `computeExtremeRegionPretransformGPD`

In [9]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_GPDmargs.csv'
res <- read_csv(path, show_col_types=FALSE)
res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


dist,n,alpha,covrate,unbounded_rate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
bivt,1000,0.01,0.862,0.566
bivt,1000,0.05,0.756,0.142
bivt,1000,0.10,0.654,0.032


**Experiment 1a2b5**
+ *gamma*: 2/3
+ $p_{n}:$ 10/n
+ *marginal estimates*: kde below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 2-way (whole sample used for marginal fitting + transformation; half goes to isoline estimation, half as a bank for bootstrap draws)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_twowaysplit.R`
+ *tube function*: `computeExtremeRegionSplit` (transformation happened inside script, outside of function)

In [10]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_split.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2e+03,0.01,0.930,0.933,0.898
1e+04,0.01,0.946,0.949,0.796
2e+04,0.01,0.940,0.943,0.772
1e+05,0.01,0.914,0.917,0.490
2e+03,0.05,0.896,0.900,0.782
1e+04,0.05,0.900,0.904,0.622
2e+04,0.05,0.888,0.892,0.566
1e+05,0.05,0.874,0.878,0.280
2e+03,0.10,0.870,0.874,0.690


**Experiment 1a2b6**
+ *gamma*: 2/3
+ $p_{n}:$ 10/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 2-way (whole sample used for marginal fitting + transformation; half goes to isoline estimation, half as a bank for bootstrap draws)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_twowaysplit_kde_margs.R`
+ *tube function*: `computeExtremeRegionSplit` (transformation happened inside script, outside function)

In [11]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_twowaysplit_kde_marg.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2000,0.01,0.954,0.957,0.910
10000,0.01,0.946,0.949,0.844
20000,0.01,0.944,0.947,0.756
2000,0.05,0.916,0.919,0.770
10000,0.05,0.914,0.917,0.662
20000,0.05,0.898,0.902,0.554
2000,0.10,0.888,0.892,0.696
10000,0.10,0.892,0.896,0.528
20000,0.10,0.888,0.892,0.408


**Experiment 1a2b7**
+ *gamma*: 2/3
+ $p_{n}:$ 10/n
+ *marginal estimates*: $(n/(n+1))\hat{F}_{ecdf}$
+ *sample splitting procedure*: 2-way (whole sample used for marginal fitting + transformation; half goes to isoline estimation, half as a bank for bootstrap draws)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_twowaysplit_rank_margs.R`
+ *tube function*: `computeExtremeRegionSplit` (transformation happened inside script, outside function)

In [12]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_twowaysplit_rank_marg.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2e+03,0.01,0.950,0.953,0.914
1e+04,0.01,0.910,0.914,0.794
2e+04,0.01,0.892,0.896,0.728
1e+05,0.01,0.842,0.846,0.438
2e+03,0.05,0.890,0.894,0.792
1e+04,0.05,0.856,0.860,0.624
2e+04,0.05,0.810,0.814,0.504
1e+05,0.05,0.754,0.758,0.242
2e+03,0.10,0.874,0.878,0.690


**Experiment 1a2b8**
+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: $(n/(n+1))\hat{F}_{ecdf}$
+ *sample splitting procedure*: 3-way (fold 1 used for marginal fitting + transformation of folds 2 and 3; fold 2 goes to isoline estimation, fold 3 goes to bank of draws for bootstrap)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threewaysplit.R`
+ *tube function*: `computeExtremeRegionThreeway`

In [13]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_split.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.924,0.927,0.890
15000,0.01,0.866,0.870,0.806
30000,0.01,0.804,0.808,0.742
150000,0.01,0.718,0.722,0.516
3000,0.05,0.844,0.848,0.752
15000,0.05,0.796,0.800,0.640
30000,0.05,0.712,0.716,0.556
150000,0.05,0.614,0.618,0.286
3000,0.10,0.820,0.824,0.686


**Experiment 1a2b9**
+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R` (script has since changed to reflect the asymmetric procedure)
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurv`

In [14]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_boot_surv.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.978,0.981,0.902
15000,0.01,0.964,0.967,0.832
30000,0.01,0.968,0.971,0.762
150000,0.01,0.958,0.961,0.506
3000,0.05,0.950,0.953,0.766
15000,0.05,0.938,0.941,0.658
30000,0.05,0.944,0.947,0.540
150000,0.05,0.924,0.927,0.254
3000,0.10,0.940,0.943,0.704


**Experiment 1a2b9.1**

A comparison to Experiment 1a2b9, using the empirical method.

+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `empirical_simulations_extreme_comp.R` (note: this script has since been repurposed for the asymmetric tubes, which is why it uses the function `computeEmpiricalRegionTwoCs`. Replacing the function, it's otherwise the same).
+ *tube function*: `computeEmpiricalRegionSymmetric`

In [15]:
path <- '~/isolines_uq/outputs/simulations/empirical_coverage_extremal_comparison.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_estimate >= p)) %>% filter(dist=='bivt') %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.990,0.993,0.176
15000,0.01,0.994,0.997,0.234
30000,0.01,0.990,0.993,0.178
150000,0.01,0.988,0.991,0.190
3000,0.05,0.966,0.969,0.000
15000,0.05,0.976,0.979,0.000
30000,0.05,0.972,0.975,0.000
150000,0.05,0.964,0.967,0.000
3000,0.10,0.936,0.939,0.000


**Experiment 1a2b9.2**

+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: no
+ *bias correction*: none
+ *simulation script*: `empirical_simulations_extreme_comp.R`
+ *tube function*: `computeEmpiricalRegionAsymmetric`

In [16]:
path <- '~/isolines_uq/outputs/simulations/empirical_coverage_extremal_comparison_twoCs.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.988,0.991,0
15000,0.01,0.994,0.997,0
30000,0.01,0.994,0.997,0
150000,0.01,0.994,0.997,0
3000,0.05,0.972,0.975,0
15000,0.05,0.972,0.975,0
30000,0.05,0.966,0.969,0
150000,0.05,0.962,0.965,0
3000,0.10,0.944,0.947,0


**Experiment 1a2b10**
+ *gamma*: 1/2
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R` (script has since changed to reflect the asymmetric procedure)
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurv`

In [17]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_boot_surv_gamma0.5.csv'
individualResults <- read_csv(path, show_col_types=FALSE)
individualResults <- individualResults %>% mutate(is_unbounded=(p <= c_estimate))
aggResults <- individualResults %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(is_unbounded)) %>% arrange(alpha)
aggResults %>% select(-c('num_covered'))

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


dist,n,alpha,covrate,unbounded_rate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
bivt,3000,0.01,0.930,0.500
bivt,15000,0.01,0.926,0.062
bivt,30000,0.01,0.872,0.006
bivt,150000,0.01,0.700,0.000
bivt,3000,0.05,0.888,0.288
bivt,15000,0.05,0.834,0.028
bivt,30000,0.05,0.794,0.000
bivt,150000,0.05,0.566,0.000
bivt,3000,0.10,0.846,0.188


**Experiment 1a2b11**
+ *gamma*: 0.6
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R` (script has since changed to reflect the asymmetric procedure)
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurv`

In [18]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_boot_surv_gamma0.6.csv'
individualResults <- read_csv(path, show_col_types=FALSE)
individualResults <- individualResults %>% mutate(is_unbounded=(p <= c_estimate))
aggResults <- individualResults %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(is_unbounded)) %>% arrange(alpha)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


In [19]:
aggResults %>% select(-c('num_covered'))

dist,n,alpha,covrate,unbounded_rate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
bivt,3000,0.01,0.956,0.812
bivt,15000,0.01,0.960,0.534
bivt,30000,0.01,0.958,0.432
bivt,150000,0.01,0.916,0.064
bivt,3000,0.05,0.906,0.624
bivt,15000,0.05,0.922,0.326
bivt,30000,0.05,0.910,0.222
bivt,150000,0.05,0.852,0.012
bivt,3000,0.10,0.878,0.508


**Experiment 1a2b12**
+ *gamma*: 2/3
+ $p_{n}:$ 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: yes
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R` (script has since changed to reflect the asymmetric procedure)
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurv`

**Note:** the thinking behind this simulation was that the empirical survival function clearly outperformed for $p_{n}=15/n$, so we are reducing to $5/n$, a situation where we have seen the empirical survival function fail.

In [20]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_boot_surv_C5.csv'
individualResults <- read_csv(path, show_col_types=FALSE)
individualResults <- individualResults %>% mutate(is_unbounded=(p <= c_estimate))
aggResults <- individualResults %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(is_unbounded)) %>% arrange(alpha)
aggResults %>% select(-c('num_covered'))

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


dist,n,alpha,covrate,unbounded_rate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
bivt,3000,0.01,0.960,0.912
bivt,15000,0.01,0.944,0.816
bivt,30000,0.01,0.934,0.720
bivt,150000,0.01,0.884,0.482
bivt,3000,0.05,0.912,0.792
bivt,15000,0.05,0.854,0.614
bivt,30000,0.05,0.860,0.536
bivt,150000,0.05,0.808,0.256
bivt,3000,0.10,0.876,0.686


**Experiment 1a2b13**
+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: no
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R` (script has since changed to reflect an update of B to 2000 instead of 1000)
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurvTwoCs`

In [21]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_boot_surv_C15_twoCs.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.910,0.914,0
15000,0.01,0.902,0.906,0
30000,0.01,0.914,0.917,0
150000,0.01,0.870,0.874,0
3000,0.05,0.836,0.840,0
15000,0.05,0.828,0.832,0
30000,0.05,0.820,0.824,0
150000,0.05,0.776,0.780,0
3000,0.10,0.772,0.776,0


**Experiment 1a2b14**
+ *gamma*: 2/3
+ $p_{n}:$ 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: 3-way (all folds used to fit and transform marginals; fold 1 used for isoline estimation, fold 2 used for bootstrap data, fold 3 used to construct survival function for final tube construction)
+ *symmetric tube?*: no
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_threeway_iso_boot_surv.R`
+ *tube function*: `computeExtremeRegionThreewayIsoBootSurvTwoCs`

**Note**: this experiment is exactly the same as 1a2b13, except $B=2000$. we thought since the asymmetric procedure now takes $\alpha/2$ and $1-\alpha/2$ quantiles, $B=1000$ would probably be a tad too low.

In [22]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_threeway_iso_2000boot_surv_C15_twoCs.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.908,0.912,0
15000,0.01,0.910,0.914,0
30000,0.01,0.872,0.876,0
150000,0.01,0.862,0.866,0
3000,0.05,0.832,0.836,0
15000,0.05,0.834,0.838,0
30000,0.05,0.782,0.786,0
150000,0.05,0.754,0.758,0
3000,0.10,0.782,0.786,0


**Experiment 1a2b15**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: none
+ *simulation script*: `extremal_simulations_asymmetric.R`
+ *tube function*: `compute_extreme_region`

**Note:** this is the same as 1a2b1, except we are doing the asymmetric procedure with $B=2000$ bootstrap replicates.

In [23]:
path <- '~/isolines_uq/outputs/simulations/extremal_coverage_asymmetric.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.900,0.904,0
5000,0.01,0.766,0.770,0
10000,0.01,0.760,0.764,0
50000,0.01,0.726,0.730,0
1000,0.05,0.838,0.842,0
5000,0.05,0.688,0.692,0
10000,0.05,0.640,0.644,0
50000,0.05,0.600,0.604,0
1000,0.10,0.792,0.796,0


**Experiment 1a2b16**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: yes; attempts to construct larger regions over which to take the supremum, exploiting the fact that regular variation bias for probability estimation is worse the sooner you make the assumption. We compute the true relative biases of the regular variation procedure over the true isoline, and use that to find worst-case oracle probability bounds, and then use that to enlarge the region we take suprema and infima over.
+ *simulation script*: `oracleregion_asymmetric_strategy1.R`
+ *tube function*: `computeExtremeRegionBC_GivenBounds`



In [24]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_true_bias_strategy1.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.886,0.890,0
5000,0.01,0.782,0.786,0
10000,0.01,0.828,0.832,0
50000,0.01,0.728,0.732,0
1000,0.05,0.818,0.822,0
5000,0.05,0.696,0.700,0
10000,0.05,0.724,0.728,0
50000,0.05,0.610,0.614,0
1000,0.10,0.768,0.772,0


**Experiment 1a2b17**
+ *gamma*: 2/3
+ $p_{n}$: 15/n
+ *marginal estimates*: ecdf below $1-n^{-\gamma}$ quantile, fitted GPD above
+ *sample splitting procedure*: threeway (all data used for marginal transformation; one fold for estimating survival function, one fold for bootstrap, and one fold for constructing enlarged sup region with oracle-computed probability bounds, detailed below)
+ *symmetric tube?*: no
+ *bias correction*: yes; attempts to construct larger regions over which to take the supremum, exploiting the fact that regular variation bias for probability estimation is worse the sooner you make the assumption. We compute the true relative biases of the regular variation procedure over the true isoline, and use that to find worst-case oracle probability bounds, and then use that to enlarge the region we take suprema and infima over. Specifically, we also bake in the transformation into this computation, sampling datasets, fitting and estimating marginals, and then only using a third of the dataset to actually get the true probability biases.
+ *simulation script*: `oracleregion_asymmetric_strategy3.2.R`
+ *tube function*: `computeExtremeRegionBCThreeway_GivenBounds`

In [25]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_oraclebias_strategy3.2.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3000,0.01,0.918,0.921,0
15000,0.01,0.902,0.906,0
30000,0.01,0.908,0.912,0
150000,0.01,0.862,0.866,0
3000,0.05,0.834,0.838,0
15000,0.05,0.830,0.834,0
30000,0.05,0.842,0.846,0
150000,0.05,0.754,0.758,0
3000,0.10,0.764,0.768,0


## 1b. Untransformed

A marginal transformation was not performed as part of the procedure.

### 2a. Oracle

Parameters for projection were assumed to be known (first and second order MRV parameters, etc.)

**Experiment 1b2a1**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *parameter assumptions*: $\xi = 1/4$
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: no
+ *simulation script*: `knownindex_nobc_asymmetric_strategy1.R` (script has since been changed to reflect gamma=1/2)
+ *tube function*: `computeExtremeRegion` (inside the main script)

In [26]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_nobc_notransform.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.
Warning message in qbeta(alpha/2, res$num_covered, 500 - res$num_covered + 1):
“NaNs produced”
Warning message in qbeta(1 - (alpha/2), res$num_covered + 1, 500 - res$num_covered):
“NaNs produced”


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,NaN,NaN,0
5000,0.01,NaN,NaN,0
10000,0.01,NaN,NaN,0
50000,0.01,NaN,NaN,0
1000,0.05,NaN,NaN,0
5000,0.05,NaN,NaN,0
10000,0.05,NaN,NaN,0
50000,0.05,NaN,NaN,0
1000,0.10,NaN,NaN,0


**Experiment 1b2a2**
+ *gamma*: 2/3
+ $p_{n}$: 5/n
+ *parameter assumptions*: $\xi = 1/4$
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: yes; use monte carlo to get true radial biases in each angular direction. Subtract biases from the estimated isoline at each angle.
+ *simulation script*: `knownindex_oracleline_asymmetric_strategy1.R` (script has since been changed to reflect gamma=1/2)
+ *tube function*: `computeExtremeRegionRadialBC` (inside the main script)

In [27]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_true_radial_bc_no_transform.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.986,0.989,0
5000,0.01,0.984,0.987,0
10000,0.01,0.992,0.995,0
50000,0.01,0.996,0.999,0
1000,0.05,0.958,0.961,0
5000,0.05,0.960,0.963,0
10000,0.05,0.982,0.985,0
50000,0.05,0.970,0.973,0
1000,0.10,0.922,0.925,0


**Experiment 1b2a3**
+ *gamma*: 1/2
+ $p_{n}$: 5/n
+ *parameter assumptions*: $\xi = 1/4$
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: none
+ *simulation script*: `knownindex_nobc_asymmetric_strategy1.R`
+ *tube function*: `computeExtremeRegion` (inside the main script)

In [28]:
# accidentally appended these results to another experiment's output
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_nobc_notransform.csv'
res <- read_csv(path, show_col_types=FALSE)
res <- res[6001:12000,]
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.720,0.724,0
5000,0.01,0.442,0.446,0
10000,0.01,0.334,0.338,0
50000,0.01,0.166,0.170,0
1000,0.05,0.516,0.520,0
5000,0.05,0.240,0.244,0
10000,0.05,0.166,0.170,0
50000,0.05,0.079,0.082,0
1000,0.10,0.424,0.428,0


**Experiment 1b2a4**
+ *gamma*: 1/2
+ $p_{n}$: 5/n
+ *parameter assumptions*: $\xi = 1/4$
+ *sample splitting procedure*: none
+ *symmetric tube?*: no
+ *bias correction*: yes; use monte carlo to get true radial biases in each angular direction. Subtract biases from the estimated isoline at each angle.
+ *simulation script*: `knownindex_oracleline_asymmetric_strategy1.R`
+ *tube function*: `computeExtremeRegionRadialBC` (inside the main script)

In [29]:
path <- '~/isolines_uq/outputs/simulations/bias_correction/extremal_coverage_true_radial_bc_no_transform_gamma_0.5.csv'
res <- read_csv(path, show_col_types=FALSE)
aggRes <- res %>% group_by(dist, n, alpha) %>% summarize(covrate=mean(is_covered), num_covered=sum(is_covered), unbounded_rate=mean(c_minus_estimate <= -p)) %>% arrange(alpha)
aggRes <- addCIs(0.95, aggRes) %>% ungroup()
aggRes %>% select(n, alpha, confint_lb, confint_ub, unbounded_rate)

`summarise()` has grouped output by 'dist', 'n'. You can override using the
`.groups` argument.


n,alpha,confint_lb,confint_ub,unbounded_rate
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1000,0.01,0.996,0.999,0
5000,0.01,0.996,0.999,0
10000,0.01,0.996,0.999,0
50000,0.01,0.994,0.997,0
1000,0.05,0.982,0.985,0
5000,0.05,0.982,0.985,0
10000,0.05,0.986,0.989,0
50000,0.05,0.980,0.983,0
1000,0.10,0.974,0.977,0


### 2b. Estimated

Parameters for projection were estimated.